In [14]:
!pip install pgmpy

In [18]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

In [19]:
model = DiscreteBayesianNetwork([
    ('Age', 'HeartDisease'),
    ('Cholesterol', 'HeartDisease'),
    ('BloodPressure', 'HeartDisease'),
    ('HeartDisease', 'ChestPain')
])

In [20]:
# Age (0 = Young, 1 = Old)
cpd_age = TabularCPD('Age', 2, [[0.6], [0.4]])

# Cholesterol (0 = Normal, 1 = High)
cpd_chol = TabularCPD('Cholesterol', 2, [[0.5], [0.5]])

# Blood Pressure (0 = Normal, 1 = High)
cpd_bp = TabularCPD('BloodPressure', 2, [[0.55], [0.45]])

# Heart Disease depends on Age, Cholesterol, BP
cpd_hd = TabularCPD(
    variable='HeartDisease',
    variable_card=2,
    values=[
        [0.9, 0.7, 0.6, 0.3, 0.8, 0.5, 0.4, 0.1],  # No Disease
        [0.1, 0.3, 0.4, 0.7, 0.2, 0.5, 0.6, 0.9]   # Disease
    ],
    evidence=['Age', 'Cholesterol', 'BloodPressure'],
    evidence_card=[2, 2, 2]
)

# Chest Pain depends on Heart Disease
cpd_cp = TabularCPD(
    variable='ChestPain',
    variable_card=2,
    values=[
        [0.8, 0.15],  # No Chest Pain
        [0.2, 0.85]   # Chest Pain
    ],
    evidence=['HeartDisease'],
    evidence_card=[2]
)

In [21]:
model.add_cpds(cpd_age, cpd_chol, cpd_bp, cpd_hd, cpd_cp)

print("Model is valid:", model.check_model())

Model is valid: True


In [22]:
inference = VariableElimination(model)

In [23]:
def get_input(question):
    val = input(question + " (yes/no): ").strip().lower()
    return 1 if val == "yes" else 0

age = get_input("Are you old?")
chol = get_input("Is your cholesterol high?")
bp = get_input("Is your blood pressure high?")
cp = get_input("Do you have chest pain?")

Are you old? (yes/no): yes
Is your cholesterol high? (yes/no): yes
Is your blood pressure high? (yes/no): yes
Do you have chest pain? (yes/no): no


In [24]:
evidence = {
    'Age': age,
    'Cholesterol': chol,
    'BloodPressure': bp,
    'ChestPain': cp
}

result = inference.query(variables=['HeartDisease'], evidence=evidence)

print("\nProbability of Heart Disease:")
print(result)


Probability of Heart Disease:
+-----------------+---------------------+
| HeartDisease    |   phi(HeartDisease) |
+=================+=====================+
| HeartDisease(0) |              0.3721 |
+-----------------+---------------------+
| HeartDisease(1) |              0.6279 |
+-----------------+---------------------+
